In [ ]:
import requests
import json
from guardian_api_key import theguardian_key
import pandas as pd
base_url = 'https://content.guardianapis.com'


In [ ]:
url = base_url  + '/sections'

params = {

    'api-key': theguardian_key
}

response = requests.get(url, params=params).json()['response']['results']

sections = []

for section in response:
    sections.append(section['webTitle'])


with open('pretty_json.json', 'w', encoding='utf-8') as f:
    json.dump(sections, f, indent=4)

sections

In [ ]:
url = base_url  + '/tags'

params = {

    'api-key': theguardian_key
}

response = requests.get(url,params=params).json()
print(response)

response = response['response']['results']

sections = []

for section in response:
    sections.append(section['webTitle'])


with open('pretty_json.json', 'w', encoding='utf-8') as f:
    json.dump(sections, f, indent=4)

sections

In [ ]:
main_df = pd.DataFrame()
queries = {
    'Apple': [
        'Apple', 'AAPL', 'iPhone', 'Tim Cook', 'App Store',
        'Apple revenue', 'MacBook', 'Apple Watch', 'Apple AI', 'antitrust Apple'
    ],
    'Exxon': [
        'Exxon', 'XOM', 'oil price', 'Darren Woods', 'fracking',
        'crude oil', 'natural gas', 'OPEC', 'carbon tax', 'Exxon acquisition'
    ],
    'Tesla': [
        'Tesla', 'TSLA', 'Elon Musk', 'electric vehicle', 'gigafactory',
        'EV subsidy', 'Tesla recall', 'autopilot', 'lithium battery', 'EV market'
    ],
    'Walmart': [
        'Walmart', 'WMT', 'retail sales', 'Doug McMillon', 'ecommerce',
        'consumer spending', 'supply chain', 'inflation retail', 'Walmart China', 'grocery sales'
    ],
    'Pfizer': [
        'Pfizer', 'PFE', 'FDA approval', 'Albert Bourla', 'drug pipeline',
        'patent expiry', 'vaccine sales', 'oncology drug', 'Pfizer acquisition', 'pharmaceutical regulation'
    ],
    'Netflix': [
        'Netflix', 'NFLX', 'streaming', 'Ted Sarandos', 'subscribers',
        'password sharing', 'Netflix ads', 'content spending', 'Disney Plus competition', 'Netflix price'
    ],
    'JPMorgan': [
        'JPMorgan', 'JPM', 'Jamie Dimon', 'interest rates', 'banking regulation',
        'Federal Reserve', 'investment banking', 'loan defaults', 'JPMorgan acquisition', 'Wall Street bonus'
    ],
    'Caterpillar': [
        'Caterpillar', 'CAT', 'infrastructure', 'Jim Umpleby', 'construction spending',
        'mining equipment', 'China construction', 'US infrastructure bill', 'heavy machinery', 'Caterpillar tariff'
    ],
}

In [ ]:
url = base_url + '/search'


for company, company_queries in queries.items():
    for query in company_queries:
        params = {
            'q': query,
            'order-by': 'relevance',
            'type': 'article',
            'section': 'business',
            'page-size':200,
            'show-fields': 'all',
            'show-tags': 'keyword',
            'api-key': theguardian_key,
            'from-date': '2021-01-01',
            'show-references': 'reuters-stock-ric', # [{'id': 'reuters-stock-ric/AMZN.O', 'type': 'reuters-stock-ric'}, {'id': 'reuters-stock-ric/AAPL.O', 'type': 'reuters-stock-ric'}, {'id': 'reuters-stock-ric/GOOG.O', 'type': 'reuters-stock-ric'}]
        }

        response = requests.get(url, params=params).json()['response']['results']
        if not response: continue # не все запросы существуют
        df = pd.DataFrame(response)
        df = pd.json_normalize(response)
        
        df['Date-cropped'] = df['webPublicationDate'].str[:10]

        def get_tags(tags):
            return ';'.join(tag['webTitle'] for tag in tags)

        def get_refs(refs):
            if len(refs) == 0: return None
            return ';'.join(ref['id'].split('/')[1].split('.')[0] for ref in refs)

        df['keywords'] = df['tags'].apply(get_tags)
        df['stocks'] = df['references'].apply(get_refs)

        df['company'] = company 
        df['search_query'] = query

        df.drop(columns=['tags','references'], inplace=True)      

        main_df = pd.concat([main_df,df], ignore_index=True)


In [ ]:
# бывает, что одинаковые статьи всплывают по разному search_query, тогда надо дропнуть


def custom_concat(x):
    return ';'.join(sorted(set(x)))

agg_df = main_df.groupby('id').agg({
    'company': custom_concat,
    'search_query': custom_concat,
}).reset_index()

dropped_df = main_df.drop(columns=['company', 'search_query']).drop_duplicates(subset=['id'])

result_df = dropped_df.merge(agg_df, on='id')

In [ ]:
result_df.drop(columns=['fields.standfirst','fields.standfirst'],inplace=True)

In [ ]:
result_df.to_csv('theguardian_raw_5.5k.csv')